In [3]:
# 5 giugno: plot degli ERD/ERS dei vari soggetti.
# ra = rest/active 
import numpy as np
import matplotlib.pyplot as plt
import mne
from mne_bids import BIDSPath, read_raw_bids
from pathlib import Path
from scipy.signal import hilbert
from scipy.ndimage import uniform_filter1d

root = Path("../data").resolve()
runs = ["4", "8", "12"]
first_person = 1
people = 10

sfreq = 160
t_pre = 1.0
t_post = 4.0
n_pre = int(t_pre * sfreq)
n_post = int(t_post * sfreq)
n_samples = n_pre + n_post
time_axis = np.linspace(-t_pre, t_post, n_samples)

channels_of_interest = ["C3", "Cz", "C4"]
bands = {"mu": (8, 12), "beta": (13, 30)}

output_dir = Path("plots/erd_ers_rest_active")
output_dir.mkdir(parents=True, exist_ok=True)


def compute_erd_ers(epochs_a, epochs_b, sfreq, n_pre, band, smooth_ms=200):
    """
    Calcola ERD/ERS% per due classi con baseline comune.
    epochs_a, epochs_b: (n_trials, n_channels, n_samples)
    """
    l_freq, h_freq = band

    def get_power(epochs):
        filt = mne.filter.filter_data(
            epochs.astype(np.float64),
            sfreq=sfreq,
            l_freq=l_freq,
            h_freq=h_freq,
            verbose=False
        )
        analytic = hilbert(filt, axis=-1)
        return np.abs(analytic) ** 2

    power_a   = get_power(epochs_a)
    power_b   = get_power(epochs_b)
    power_all = get_power(np.concatenate([epochs_a, epochs_b], axis=0))

    smooth_samples  = int(smooth_ms * sfreq / 1000)
    baseline_common = np.mean(power_all[:, :, :n_pre], axis=(0, 2))[:, np.newaxis]

    mean_a = uniform_filter1d(np.mean(power_a, axis=0), size=smooth_samples, axis=-1)
    mean_b = uniform_filter1d(np.mean(power_b, axis=0), size=smooth_samples, axis=-1)

    erd_a = (mean_a - baseline_common) / baseline_common * 100
    erd_b = (mean_b - baseline_common) / baseline_common * 100

    return erd_a, erd_b


def plot_erd_ers(erd_a, erd_b, label_a, label_b, color_a, color_b,
                 channels, bands, time_axis, title, save_path):
    fig, axes = plt.subplots(
        len(bands), len(channels),
        figsize=(14, 8),
        sharex=True
    )
    fig.suptitle(title, fontsize=14)

    for row, band_name in enumerate(bands.keys()):
        for col, ch_name in enumerate(channels):
            ax = axes[row, col]
            ax.plot(time_axis, erd_a[band_name][col], color=color_a, label=label_a, linewidth=1.5)
            ax.plot(time_axis, erd_b[band_name][col], color=color_b, label=label_b, linewidth=1.5)
            ax.axvline(x=0,   color='black', linestyle='--', linewidth=1)
            ax.axhline(y=0,   color='gray',  linestyle=':',  linewidth=0.8)
            ax.axvspan(-t_pre, 0, alpha=0.08, color='gray')
            ax.set_title(f"{band_name.capitalize()} – {ch_name}")
            ax.set_ylabel("ERD/ERS (%)")
            ax.legend(fontsize=7)
            ax.grid(True, alpha=0.3)

    axes[-1, 1].set_xlabel("Tempo (s)")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()


# Accumulo per plot medio
all_erd_rest   = {band: [] for band in bands}
all_erd_active = {band: [] for band in bands}

for i in range(first_person, first_person + people):
    subject = f"{i:03d}"
    epochs_rest   = []
    epochs_active = []

    for run in runs:
        bids_path = BIDSPath(
            subject=subject,
            task="motion",
            run=run,
            datatype="eeg",
            root=root,
        )

        try:
            raw = read_raw_bids(bids_path, verbose=False)
            raw.load_data(verbose=False)
            raw.filter(l_freq=1, h_freq=40, verbose=False)
            raw.set_eeg_reference('average', projection=False, verbose=False)

            events, event_id = mne.events_from_annotations(raw, verbose=False)

            # T0 = rest, T1 e T2 = active (aggregati)
            event_map = {
                event_id['TASK2T0']: 'rest',
                event_id['TASK2T1']: 'active',
                event_id['TASK2T2']: 'active',
            }

            picks = mne.pick_channels(raw.ch_names, channels_of_interest)
            data  = raw.get_data(picks=picks)

            for event in events:
                onset_sample = event[0]
                event_code   = event[2]

                if event_code not in event_map:
                    continue

                label = event_map[event_code]
                start = onset_sample - n_pre
                end   = onset_sample + n_post

                if start < 0 or end > data.shape[1]:
                    continue

                epoch = data[:, start:end]
                if label == 'rest':
                    epochs_rest.append(epoch)
                else:
                    epochs_active.append(epoch)

        except Exception as e:
            print(f"Errore soggetto {subject} run {run}: {e}")

    if not epochs_rest or not epochs_active:
        print(f"Soggetto {subject}: trial insufficienti, skip")
        continue

    X_rest   = np.array(epochs_rest)
    X_active = np.array(epochs_active)
    print(f"Soggetto {subject} - Trial rest: {len(X_rest)}, Trial active: {len(X_active)}")

    erd_rest_subj   = {}
    erd_active_subj = {}

    for band_name, band_range in bands.items():
        erd_r, erd_a = compute_erd_ers(X_rest, X_active, sfreq, n_pre, band_range)
        erd_rest_subj[band_name]   = erd_r
        erd_active_subj[band_name] = erd_a
        all_erd_rest[band_name].append(erd_r)
        all_erd_active[band_name].append(erd_a)

    plot_erd_ers(
        erd_rest_subj, erd_active_subj,
        label_a='Rest', label_b='Active',
        color_a='green', color_b='orange',
        channels=channels_of_interest,
        bands=bands,
        time_axis=time_axis,
        title=f"ERD/ERS Rest vs Active – Soggetto {subject}",
        save_path=output_dir / f"subject_{subject}.png"
    )
    print(f"Soggetto {subject} salvato")

# Plot medio
mean_erd_rest   = {band: np.mean(all_erd_rest[band],   axis=0) for band in bands}
mean_erd_active = {band: np.mean(all_erd_active[band], axis=0) for band in bands}

plot_erd_ers(
    mean_erd_rest, mean_erd_active,
    label_a='Rest', label_b='Active',
    color_a='green', color_b='orange',
    channels=channels_of_interest,
    bands=bands,
    time_axis=time_axis,
    title="ERD/ERS Rest vs Active – Media tutti i soggetti",
    save_path= output_dir/"mean_all_subjects.png"
)
print("Plot medio salvato")

Soggetto 001 - Trial rest: 42, Trial active: 45
Soggetto 001 salvato
Soggetto 002 - Trial rest: 42, Trial active: 45
Soggetto 002 salvato
Soggetto 003 - Trial rest: 42, Trial active: 45
Soggetto 003 salvato
Soggetto 004 - Trial rest: 42, Trial active: 45
Soggetto 004 salvato
Soggetto 005 - Trial rest: 42, Trial active: 45
Soggetto 005 salvato
Soggetto 006 - Trial rest: 42, Trial active: 45
Soggetto 006 salvato
Soggetto 007 - Trial rest: 42, Trial active: 45
Soggetto 007 salvato
Soggetto 008 - Trial rest: 42, Trial active: 45
Soggetto 008 salvato
Soggetto 009 - Trial rest: 42, Trial active: 45
Soggetto 009 salvato
Soggetto 010 - Trial rest: 42, Trial active: 45
Soggetto 010 salvato
Plot medio salvato
